# QEC 513 Delay Benchmark

Standalone fidelity benchmark for the five-qubit code. The notebook compares a noise-free Aer reference to optional IBM hardware delay sweeps and saves QEC benchmark JSON files under `results/qec513/`.


In [ ]:
from datetime import datetime
import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from qiskit.transpiler import generate_preset_pass_manager
from qiskit_aer.primitives import SamplerV2 as AerSampler
from qiskit_ibm_runtime import QiskitRuntimeService, SamplerV2 as Sampler

from broadcasting import qec_513_delay_benchmark_circuit
from broadcasting.plotting import save_figure

%matplotlib inline
plt.rcParams.update({"figure.dpi": 120})


## Configuration


In [ ]:
USE_QEC = True
RUN_HARDWARE = False
SAVE_FIGURE = True

seed = 42
rng = np.random.default_rng(seed)
theta = float(rng.uniform(0, np.pi))
phi = float(rng.uniform(0, 2 * np.pi))
tau_values = np.linspace(0, 6000, 21).astype(int).tolist()
shots = 8192

IBM_PROFILE = "mprest1"
IBM_BACKEND = "ibm_kingston"
OPTIMIZATION_LEVEL = 0

QEC_RESULTS_DIR = Path("results/qec513")
QEC_RESULTS_DIR.mkdir(parents=True, exist_ok=True)
FIGURE_DIR = Path("figures")

print(f"USE_QEC={USE_QEC}  theta={theta:.4f}  phi={phi:.4f}")
print(f"tau values: {tau_values[0]}..{tau_values[-1]} dt ({len(tau_values)} points)")


## Build Circuit


In [ ]:
qc, tau_param, fid_reg = qec_513_delay_benchmark_circuit(
    theta,
    phi,
    use_qec=USE_QEC,
)
print(f"Circuit depth: {qc.depth()}")
print(f"Qubits: {qc.num_qubits}")
qc.draw("mpl", fold=120)


## Noise-Free Reference


In [ ]:
def fidelity_from_pub(pub_result, register_name):
    counts = getattr(pub_result.data, register_name).get_counts()
    total = sum(counts.values())
    return sum(v for bitstring, v in counts.items() if bitstring[-1] == "0") / total

bound_circuits = [qc.assign_parameters({tau_param: int(t)}) for t in tau_values]
aer_result = AerSampler().run([(c,) for c in bound_circuits], shots=shots).result()
ideal_fidelities = [fidelity_from_pub(pub, fid_reg) for pub in aer_result]
print(f"Noise-free mean fidelity: {np.mean(ideal_fidelities):.4f}")


## Optional Hardware Sweep


In [ ]:
backend_fidelities = []
backend_counts = []
qec_outfile = None

if RUN_HARDWARE:
    service = QiskitRuntimeService(name=IBM_PROFILE)
    backend = service.backend(name=IBM_BACKEND)
    print(f"Backend: {backend.name}, dt={backend.dt} seconds")

    pass_manager = generate_preset_pass_manager(
        backend=backend,
        optimization_level=OPTIMIZATION_LEVEL,
    )
    isa_circuit = pass_manager.run(qc)
    isa_circuits = [isa_circuit.assign_parameters({tau_param: int(t)}) for t in tau_values]

    job = Sampler(mode=backend).run([(c,) for c in isa_circuits], shots=shots)
    print(f"Job ID: {job.job_id()}")
    hardware_result = job.result()

    for pub in hardware_result:
        counts = getattr(pub.data, fid_reg).get_counts()
        total = sum(counts.values())
        backend_fidelities.append(
            sum(v for bitstring, v in counts.items() if bitstring[-1] == "0") / total
        )
        backend_counts.append(dict(counts))

    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    qec_outfile = QEC_RESULTS_DIR / f"qec513_delay_sweep_{timestamp}.json"
    with open(qec_outfile, "w") as f:
        json.dump(
            {
                "timestamp": datetime.now().isoformat(),
                "experiment": "qec_513_delay_sweep",
                "job_id": job.job_id(),
                "optimization_level": OPTIMIZATION_LEVEL,
                "backend": backend.name,
                "shots": shots,
                "seed": seed,
                "use_qec": USE_QEC,
                "state_prep": {"theta": theta, "phi": phi},
                "tau_values": [int(t) for t in tau_values],
                "ideal_fidelities": [float(f) for f in ideal_fidelities],
                "backend_fidelities": [float(f) for f in backend_fidelities],
                "backend_counts": backend_counts,
            },
            f,
            indent=2,
        )
    print(f"Saved hardware sweep to {qec_outfile}")
else:
    print("Set RUN_HARDWARE=True to submit the benchmark to IBM hardware.")


## Plot Current Sweep


In [ ]:
x_us = 4e-3 * np.asarray(tau_values, dtype=float)
fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(x_us, ideal_fidelities, "o-", color="tab:green", label="Noise-free")
if backend_fidelities:
    ax.plot(
        x_us,
        backend_fidelities,
        "s-",
        color="tab:red",
        label=f"{IBM_BACKEND} opt={OPTIMIZATION_LEVEL}",
    )
ax.axhline(0.5, color="gray", linestyle="--", alpha=0.5, label="Random (0.5)")
ax.set_xlabel("Delay time (us)")
ax.set_ylabel("Fidelity")
ax.set_ylim(0, 1.05)
ax.grid(alpha=0.25)
ax.legend()
plt.tight_layout()

if SAVE_FIGURE:
    suffix = "qec" if USE_QEC else "no_qec"
    figure_path = FIGURE_DIR / f"qec513_delay_sweep_{suffix}.png"
    save_figure(fig, figure_path)
    print(f"Saved title-free figure to {figure_path}")
plt.show()


## Compare Saved QEC Sweeps


In [ ]:
qec_files = sorted(QEC_RESULTS_DIR.glob("qec513_delay_sweep_*.json"))
if not qec_files:
    print("No saved QEC sweeps found.")
else:
    saved = []
    seen_job_ids = {}
    for path in qec_files:
        with open(path) as f:
            record = json.load(f)
        job_id = record.get("job_id")
        if job_id is not None and job_id in seen_job_ids:
            print(f"{path.name}: SKIPPED (duplicate job_id={job_id}, same as {seen_job_ids[job_id]})")
            continue
        if job_id is not None:
            seen_job_ids[job_id] = path.name
        saved.append(record)
        use_qec_label = record.get("use_qec")
        use_qec_str = "unknown" if use_qec_label is None else ("QEC" if use_qec_label else "No QEC")
        print(
            f"{path.name}: backend={record.get('backend')} "
            f"opt={record.get('optimization_level')} use_qec={use_qec_str}"
        )

    fig, ax = plt.subplots(figsize=(8, 5))
    reference = saved[0]
    x_ref = 4e-3 * np.asarray(reference["tau_values"], dtype=float)
    ax.plot(x_ref, reference["ideal_fidelities"], color="tab:green", label="Noise-free")

    for run in saved:
        tau = 4e-3 * np.asarray(run["tau_values"], dtype=float)
        use_qec_label = run.get("use_qec")
        # Do not assume True for missing use_qec -- it was never recorded for the
        # six legacy files here, and defaulting to True silently mislabels them.
        if use_qec_label is None:
            qec_str = "QEC?"
        else:
            qec_str = "QEC" if use_qec_label else "No QEC"
        label = f"{qec_str} opt={run.get('optimization_level', '?')}"
        ax.plot(tau, run["backend_fidelities"], linewidth=1.2, label=label)

    ax.axhline(0.5, color="gray", linestyle="--", alpha=0.5)
    ax.set_xlabel("Delay time (us)")
    ax.set_ylabel("Fidelity")
    ax.set_ylim(0, 1.05)
    ax.grid(alpha=0.25)
    ax.legend()
    plt.tight_layout()
    if SAVE_FIGURE:
        save_figure(fig, FIGURE_DIR / "qec513_saved_sweeps.png")
    plt.show()
